# Parsing log files for Grid search loss

In [1]:
import pandas as pd
import re

def parse_experiment_logs(log_file_path: str) -> pd.DataFrame:
    """
    Parses the grid search log file to extract loss configurations and the best
    metrics for each run using a robust, single-pass parsing strategy.

    Args:
        log_file_path (str): The path to the log file.

    Returns:
        pd.DataFrame: A DataFrame containing the parsed results.
    """
    with open(log_file_path, 'r') as f:
        log_content = f.read()

    # --- Step 1: Isolate the final experiment run ---
    experiment_start_pattern = "============  STARTING EXPERIMENT STF_HC  ============"
    last_start_index = log_content.rfind(experiment_start_pattern)
    if last_start_index == -1:
        print("Experiment start pattern not found.")
        return pd.DataFrame()

    experiment_logs = log_content[last_start_index:]

    # --- Step 2: Define Regex patterns ---
    # Pattern to find a complete run block, from "Starting Run" to the start of the next one (or end of file)
    run_block_pattern = re.compile(
        r"INFO - \n=+\n.*?Starting Run \d+/\d+ \| Loss Config ID: (cfg_\d+)" # Start of a block
        r"(.*?)" # Capture everything inside the block (non-greedy)
        r"(?=INFO - \n=+\n.*?Starting Run|\Z)", # Positive lookahead for the next block or end of string
        re.DOTALL
    )

    config_pattern = re.compile(
        r"Loss Config \((cfg_\d+)\):"
        r".*?LINK\s+=\s+([\d\.]+)"
        r".*?ENTROPY\s+=\s+([\d\.]+)"
        r".*?RECONSTRUCTION\s+=\s+([\d\.]+)"
        r".*?CONTRASTIVE\s+=\s+([\d\.]+)"
        r".*?BALANCE\s+=\s+([\d\.]+)"
        r".*?REPEL\s+=\s+([\d\.]+)", re.DOTALL
    )

    epoch_metrics_pattern = re.compile(
        r"Epoch (\d+) \| F1: ([\d\.]+) \| Comp: ([\d\.]+) \| Conf: ([\d\.]+) \| Mod: ([\d\.]+) \| Sil: ([\d\.]+) \| E-Score: ([\d\.]+)"
    )

    save_pattern = re.compile(r"New best model saved with E-Score:")

    # --- Step 3: Extract data for each configuration ---
    results = []
    # Find all non-overlapping matches of the entire run block
    all_runs = run_block_pattern.findall(experiment_logs)

    for config_id_from_start, run_log in all_runs:
        config_match = config_pattern.search(run_log)
        if not config_match:
            continue

        _, link, entropy, recon, contrast, balance, repel = config_match.groups()

        # Find the final best metrics for this run
        best_metrics_for_run = None
        lines = run_log.strip().split('\n')
        for idx, line in enumerate(lines):
            if save_pattern.search(line):
                # The metrics are in the previous line
                if idx > 0:
                    prev_line = lines[idx - 1]
                    metrics_match = epoch_metrics_pattern.search(prev_line)
                    if metrics_match:
                        epoch, f1, comp, conf, mod, sil, escore = metrics_match.groups()
                        best_metrics_for_run = {
                            "Best Epoch": int(epoch),
                            "F1": float(f1),
                            "Completeness": float(comp),
                            "Conformity": float(conf),
                            "Modularity": float(mod),
                            "Silhouette": float(sil),
                            "E-Score": float(escore),
                        }

        if not best_metrics_for_run:
            best_metrics_for_run = {
                "Best Epoch": "N/A", "F1": 0.0, "Completeness": 0.0,
                "Conformity": 0.0, "Modularity": 0.0, "Silhouette": 0.0,
                "E-Score": 0.0
            }

        row = {
            "Config ID": config_id_from_start,
            "Link Loss": float(link),
            "Entropy Loss": float(entropy),
            "Recon Loss": float(recon),
            "Contrastive Loss": float(contrast),
            "Balance Loss": float(balance),
            "Repel Loss": float(repel),
            **best_metrics_for_run
        }
        results.append(row)

    df = pd.DataFrame(results)
    return df

In [2]:
log_file = '../logs/training_model_experiment_gridsearch_STF_HC_loss.log'
results_df = parse_experiment_logs(log_file)


if not results_df.empty:
    # Sort by the E-Score to easily see the best configurations
    results_df = results_df.sort_values(by="E-Score", ascending=False).reset_index(drop=True)

    # Calculate HI-Score for the sorted results
    results_df['HI-Score'] = (results_df['Completeness'] * results_df['Conformity'] * results_df['Modularity'] * results_df['Silhouette']) ** 0.25

    output_filename = 'log_analysis_results.xlsx'
    results_df.to_excel(output_filename, index=False, float_format="%.4f")

    print(f"Successfully parsed {len(results_df)} configurations.")
    print(f"Results exported to '{output_filename}'")
    print("\n--- Top 5 Configurations by E-Score ---")
    print(results_df.head(5).to_string())

Successfully parsed 101 configurations.
Results exported to 'log_analysis_results.xlsx'

--- Top 5 Configurations by E-Score ---
   Config ID  Link Loss  Entropy Loss  Recon Loss  Contrastive Loss  Balance Loss  Repel Loss  Best Epoch      F1  Completeness  Conformity  Modularity  Silhouette  E-Score  HI-Score
0  cfg_00000        0.0          0.00        0.00              0.00          0.00        0.00           1  1.0000           1.0      0.5745      0.3518      0.4142   0.6995  0.537897
1  cfg_00082       10.0          0.01        0.01              0.01          0.01        0.01           4  0.4898           1.0      0.1559      0.3510      0.4711   0.4408  0.400697
2  cfg_00368       10.0          0.10        0.10              0.10          1.00        0.10          15  1.0000           1.0      0.0239      0.3653      0.4803   0.4056  0.254472
3  cfg_00301        1.0          0.10        1.00              0.01          0.10        0.01          10  1.0000           1.0      0.0222

In [3]:
results_df

,Config ID,Link Loss,Entropy Loss,Recon Loss,Contrastive Loss,Balance Loss,Repel Loss,Best Epoch,F1,Completeness,Conformity,Modularity,Silhouette,E-Score,HI-Score
0,cfg_00000,0.0,0.00,0.00,0.00,0.00,0.00,1,1.0000,1.0,0.5745,0.3518,0.4142,0.6995,0.537897
1,cfg_00082,10.0,0.01,0.01,0.01,0.01,0.01,4,0.4898,1.0,0.1559,0.3510,0.4711,0.4408,0.400697
2,cfg_00368,10.0,0.10,0.10,0.10,1.00,0.10,15,1.0000,1.0,0.0239,0.3653,0.4803,0.4056,0.254472
3,cfg_00301,1.0,0.10,1.00,0.01,0.10,0.01,10,1.0000,1.0,0.0222,0.3251,0.4782,0.3902,0.242379
4,cfg_00430,100.0,0.10,0.01,1.00,1.00,0.01,19,0.4846,1.0,0.0684,0.3398,0.4693,0.3878,0.323171
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,cfg_00225,100.0,0.01,1.00,0.01,1.00,1.00,1,0.3537,1.0,0.0000,0.3468,0.4820,0.0000,0.000000
97,cfg_00221,100.0,0.01,1.00,0.01,0.10,0.10,1,0.4385,1.0,0.0000,0.3517,0.4821,0.0000,0.000000
98,cfg_00215,100.0,0.01,0.10,1.00,1.00,0.10,1,0.4088,1.0,0.0000,0.3430,0.4802,0.0000,0.000000
99,cfg_00204,100.0,0.01,0.10,0.10,0.10,1.00,1,0.0780,1.0,0.0000,0.3461,0.4818,0.0000,0.000000
